## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
2. What is a confusion table? What does it help us understand about a model's performance?
3. What does the SSE quantify about a particular model?
4. What are overfitting and underfitting?
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Arya Kumar

**Date:** 10/02/26

In [ ]:
# Your Phase 1 solution to the problem goes here.


Question 1.1: Regression models predict continous values like house prices. Classification models predict discrete categories like whether an email is spam or not.

Question 1.2: A confusion matrix is a visualization method for classifier algorithm results. Confusion matrices help us evaluate the performance of a classification model and can help evaluate model performance metrics like precision and recall. The grid structure of confusion matrices help visualize model classification accuracy by displaying the number of correct predictions and incorrect predictions for all classes alongside one another.

Question 1.3: The Sum of Squared Errors is a measure in a regression analysis that quantifies how well a model fits the data by measuring the total squared differences between observed and predicted values.

Question 1.4: Underfitting is when an algorithim is too simple to capture some of the patterns in the data. Overfitting is when the algorithim is too complex and misreads the patterns in the date and treats random noise in the training data as important data points

Question 1.5: The training set is helpful for training the model while testing set can be used to evaluate the model's performance after training. Essentially we don't want to test the model on the same set of data we used to train it otherwise it could just be memorizing the training data and it would not actually work on new data. Choosing the right K helps to balance bias and variance and prevent overfitting or underfitting. Measuring accuracy or SSE across different values of k reveals the best spot where the model stops reducing error due to noise and starts predicting genuine patterns accurately.

Question 1.6: Reporting a class label as a prediction can lead to a straightfoward binary like for example yes or no. It's easy to interpret and useful for simple decision-making. However, it does not indicate how confident the model is that its classification is correct. For example, it would be like if we are taking a flu test, we would get back a simple positive that you have the flu. But the test does not tell you how confident they are that you have the flu on a percentage basis. Reporting a class label as a probability distribution would let us know that the test is 55% positive is therefore is not overwhelmingly certain. This makes it a little more complicated as you may have to set thresholds and it may also be harder to interpret the percentage number.

Question 2.1

In [1]:
from google.colab import files
uploaded = files.upload()

Saving land_mines.csv to land_mines.csv


In [4]:
import pandas as pd
df = pd.read_csv("land_mines.csv")

df.head()

,voltage,height,soil,mine_type
0,0.338157,0.000000,0.0,1
1,0.320241,0.181818,0.0,1
2,0.287009,0.272727,0.0,1
3,0.256284,0.454545,0.0,1
4,0.262840,0.545455,0.0,1


In [11]:
#Checking info of dataframe - no missing variables and there are four columns and 338 rows. everythnig except mine_type is a float type.
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB


,0
voltage,0
height,0
soil,0
mine_type,0


In [13]:
#investigating the mine_type variable - appears fairly balanced with a similar number of mines in each category type
df["mine_type"].value_counts()

,count
mine_type,
1,71
2,70
3,66
4,66
5,65


In [16]:
#investigating the other categories - The three features are voltage, height, and soil, all of which are numeric and scaled between 0 and 1.
df[["voltage", "height", "soil"]].describe()

,voltage,height,soil
count,338.000000,338.000000,338.000000
mean,0.430634,0.508876,0.503550
std,0.195819,0.306043,0.344244
min,0.197734,0.000000,0.000000
25%,0.309737,0.272727,0.200000
50%,0.359516,0.545455,0.600000
75%,0.482628,0.727273,0.800000
max,0.999999,1.000000,1.000000


In [17]:
df[["voltage", "height", "soil"]].nunique() #many different types of voltages with very little variation in types of soil and height across 338 entries. Voltage appears to be a much more continous variable compared to height and soil

,0
voltage,196
height,12
soil,6


In [19]:
#2.2 - importing and defining x as the features and y as the mine type to set up for split. selected a random state in orer to make my results reproducibel by ensuring the randomization is always teh same to reproduce results. testsize as 0.5 splits the data half and half.
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]

y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=37
)


In [21]:
#2.3 - testing k values from 1-30 and seeing the corresponding accuracy to evaluate which k is optimal - it seems like K=3 is the most accurate as it appears to peak at 0.3905. This means the accuracy of the k=3 is approximately 39.1%.
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

for k in range(1, 31):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(k, accuracy)

1 0.31952662721893493
2 0.3727810650887574
3 0.3905325443786982
4 0.3668639053254438
5 0.3254437869822485
6 0.33727810650887574
7 0.33136094674556216
8 0.3431952662721893
9 0.34911242603550297
10 0.33136094674556216
11 0.33136094674556216
12 0.3431952662721893
13 0.35502958579881655
14 0.35502958579881655
15 0.3431952662721893
16 0.3668639053254438
17 0.3609467455621302
18 0.34911242603550297
19 0.33727810650887574
20 0.3431952662721893
21 0.33727810650887574
22 0.31952662721893493
23 0.33136094674556216
24 0.3136094674556213
25 0.3136094674556213
26 0.3136094674556213
27 0.30177514792899407
28 0.3076923076923077
29 0.3076923076923077
30 0.30177514792899407


In [24]:
#2.4 -  using the optimal k=3 to build the knn model and to compare its predicted mine types to the actual mine types in the test set. We get an array where to columns are predicted mine type and rows is actual mine type.
from sklearn.metrics import confusion_matrix, accuracy_score

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

confusion_matrix(y_test, y_pred)

array([[22,  0,  6,  3,  2],
       [ 0, 28,  3,  0,  3],
       [12,  2,  8,  4,  2],
       [14,  4,  9,  6,  2],
       [17,  0, 15,  5,  2]])

2.4 continued - Looking at diaganols above helps us understand model performance for the different mine types. The model performs best for types 1 and 2 where there is 22 and 28 correct respectively. It performs much worse for type 5 where there is 2 correct and relatively worse for types 3 and 4 where tehre is 8 and 6 correct respectively. We can actually see for type 5 how it is most often misclassified as type one or three at 17 and 15 respectively.

In [23]:
accuracy_score(y_test, y_pred) - #overall accuracy is 39.05% as we knew from 2.3

0.3905325443786982

2.5 I would not recommend using this model as the sole method for identifying a mine because its overall accuracy is only about 39%. It could be used as a supplementary confirmation for type one and two where there is relatively higher accuracy, but the model accuracy itself is still too low to be used as the sole basis for mine types.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [b10688]
- **AI tool and model used:** [Chat gpt 5.6 sol]

### *Prompts used

1. Do my reasonings make sense for Question 1.1 - 1.6?
2. Does my code for question two make sense and can it be improved?

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. Improvement for question 1.6 - Your explanation correctly says probabilities are harder to interpret and require thresholds, but you should mention calibration: a model reporting 80% probability does not automatically mean predictions with 80% probability are actually correct 80% of the time.
2. Q2.1 could use one actual visualization. Your EDA currently has info(), missing values, class counts, describe(), and nunique(), which is solid. But because the prompt specifically says EDA, I would add a simple histogram of the three features or a bar chart of mine_type. That makes the exploration more complete.
3. Q2.2: consider stratify=y. Your professor explicitly warns about classes accidentally becoming unevenly divided between training and testing.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | **Accept** / modify / reject | I accept it because it is a valid way of improving my explanation and making what I was explaining as the weakness more clear.
| 2 | **Accept** / modify / reject | I accept this too. It could have an actual visualization.
| 3 | **Accept** / modify / reject | I accept this too. I did not know about the stratify = y part and it seems like a good thing to add to prevent the problem it discusses.

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.

Question 1.6 - Reporting a class label is straightforward and gives a clear prediction, but it does not show how confident the model is in that prediction. A probability distribution provides more information by showing the model’s confidence across possible classes, but it can be harder to interpret and may require choosing a probability threshold to make a final classification. Additionally, the probabilities may not always be well calibrated, meaning a predicted probability of 80% does not necessarily mean the model will be correct 80% of the time.

In [ ]:
# 2.1 - using my same explanation that is above, but with this improved code to include the visualization.

import pandas as pd
import matplotlib.pyplot as plt

# Load the land mines data
df = pd.read_csv("land_mines.csv")

# Look at the first few rows
df.head()

# Check the number of observations, variable types, and missing values
df.info()

# Check for missing values in each variable
df.isna().sum()

# Summarize the distribution of the target variable
df["mine_type"].value_counts().sort_index()

# Graph the distribution of the target variable
df["mine_type"].value_counts().sort_index().plot(kind="bar")

plt.xlabel("Mine Type")
plt.ylabel("Count")
plt.title("Distribution of Mine Types")
plt.show()

# Summary statistics for the three features
df[["voltage", "height", "soil"]].describe()

df[["voltage", "height", "soil"]].hist()

plt.tight_layout()
plt.show()

df[["voltage", "height", "soil"]].nunique()

In [ ]:
#question 2.2
from sklearn.model_selection import train_test_split

X = df[["voltage", "height", "soil"]]

y = df["mine_type"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.5, random_state=37, stratify=y
)

### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

AI was helpful in this assignment. It first helped me improve my explanation for the strengths/weaknesses of putting a class label as a prediction vs probability distributions. I had strengths for both, but my weakness for probability distribution was not fully fleshed out. The AI answer was able to draw out a more clear explanation for why the percentage answer could be misintrepreted easily and that therefore makes probability distributions a more complex answer.

For my code, it helped introduce the stratify concept for question 2.2. I did not originally know what the stratify part meant, but chat explained that my original code randomly divided the data 50/50, but the prompt specifically warned that classes could become unevenly distributed between the training and test sets. using stratify = y directly addresses this problem by keeping the proportions of the five mine types approximately equal across the two sets.

AI additionally helped with question 2.1 where I did not have a visualization for my EDA. I did some good parts of EDA by having some describe and info functsions and commenst explaining what I saw for the features and target, but an additional visualization for teh features would have been a good aspect of EDA.